# Using CLARITY

## Goal

Load the released Spider/BIRD conversations, inspect an ambiguity example and its
schema, rank candidate columns, and prepare a small generation run.

**Run All works offline after setup.** Live generation is disabled by default.
This walkthrough uses CLARITY 0.2.1's Python API and the included benchmark files.

[Paper](https://aclanthology.org/2026.acl-industry.86/) ·
[Code](https://github.com/duyvuleo/clarity) ·
[Data card](../DATA_CARD.md) · [Configuration guide](../SCHEMA_RETRIEVAL.md)

## Setup

Use Python 3.11 or later; this notebook is checked with Python 3.12.
In a terminal, clone the repository and install CLARITY and Jupyter:

```bash
git clone https://github.com/duyvuleo/clarity.git
cd clarity
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements-lock.txt
python -m pip install --no-deps -e .
python -m pip install 'jupyterlab>=4,<5' 'ipykernel>=6,<8'
python -m jupyterlab notebooks/CLARITY_quickstart.ipynb
```

Select the Python kernel from this environment. The repository or main release
ZIP supplies `data/`; installing the software wheel alone does not supply it.
The offline cells need no model credentials, SQLite databases or optional value catalogs.

In [1]:
import copy
import hashlib
import json
import os
from collections import Counter
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import display

import clarity
from clarity.datasets import DATASETS, load_annotations, validate_dataset
from clarity.utils.file_io import load_json, load_jsonl

# Launch from the repository root or a directory below it, including notebooks/.
ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "configs/spider_lexical.json").is_file()
        and (path / "data/spider/annotations.jsonl").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the CLARITY repository or main release ZIP.")

DATA_ROOT = ROOT / "data"
DATASET = "spider"  # Change to "bird" and rerun from here.
RUN_LIVE_GENERATION = False
if DATASET not in DATASETS:
    raise ValueError("DATASET must be spider or bird.")

print(f"CLARITY {clarity.__version__}; selected dataset: {DATASET}")

CLARITY 0.2.1; selected dataset: spider


## Steps

### 1. Validate and summarize the benchmark

`validate_dataset` checks required fields, conversation roles and database IDs.
Its counts describe the packaged files. This structural check does not judge
whether every generated utterance is semantically correct.

In [2]:
validation = [validate_dataset(DATA_ROOT, dataset) for dataset in DATASETS]
summary = pd.DataFrame(validation)[["dataset", "annotations", "seeds", "databases"]]
display(summary.set_index("dataset"))

# Equivalent terminal command: clarity validate --data-root data
assert summary["annotations"].sum() == 19_268

,annotations,seeds,databases
dataset,,,
spider,12098,7000,140
bird,7170,1534,11


### 2. Load and filter conversations

Each annotation is a dictionary. `usecase` combines the ambiguity/confusion
setting, number of facets, and conversation scenario. Historical **confusion**
labels denote unanswerability settings. The two historical "not helpful"
variants share a label, so these labels are not unique conversation identifiers.

In [3]:
annotations = load_annotations(DATA_ROOT, DATASET)
usecase_counts = Counter(row["usecase"] for row in annotations)
display(
    pd.DataFrame(usecase_counts.most_common(8), columns=["usecase", "conversations"]).set_index(
        "usecase"
    )
)

selected_usecase = "lexical column ambiguity single - concise helpful conversation type"
selected = [row for row in annotations if row["usecase"] == selected_usecase]
if not selected:
    raise ValueError("Choose a usecase present in this dataset.")
example = selected[0]
print(f"Selected {len(selected):,} conversations; showing the first in file order.")

,conversations
usecase,
column confusion single - conversation type not applicable,1859
value confusion single - conversation type not applicable,1456
column confusion multiple - conversation type not applicable,1360
semantic column ambiguity single - not helpful conversation type,1224
lexical column ambiguity single - not helpful conversation type,980
semantic column ambiguity single - concise helpful conversation type,612
semantic column ambiguity single - partially helpful conversation type,612
semantic column ambiguity single - verbose helpful conversation type,612


Selected 490 conversations; showing the first in file order.


### 3. Read one example and its dialogue

`question` is the generated request; `original_question` is its parent request.
`selected_column`, `ac_term` and `similar_columns` describe the generated pivot
and candidate interpretations. `expectedQuery` is inherited **parent SQL**;
it is not a verified terminal SQL target for every dialogue.

In [4]:
with pd.option_context("display.max_colwidth", None):
    display(
        pd.DataFrame(
            [
                {"field": field, "value": example.get(field)}
                for field in [
                    "db_id",
                    "usecase",
                    "question",
                    "original_question",
                    "selected_column",
                    "ac_term",
                    "similar_columns",
                    "expectedQuery",
                ]
            ]
        ).set_index("field")
    )
    display(pd.DataFrame(example["conversation"])[["role", "content"]])

,value
field,
db_id,farm
usecase,lexical column ambiguity single - concise helpful conversation type
question,List horses sorted in ascending order.
original_question,List the total number of horses on farms in ascending order.
selected_column,[Total_Horses]
ac_term,[horses]
similar_columns,"[Total_Horses, working_horses]"
expectedQuery,SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC


,role,content
0,USER,List horses sorted in ascending order.
1,AGENT,"Just to confirm—when you say 'horses,' do you mean 'working_horses' or 'Total_Horses'?"
2,USER,Total_Horses
3,AGENT,SELECT Total_Horses FROM farm ORDER BY Total_Horses ASC


### 4. Inspect the source schema

Schemas and column catalogs are keyed by database ID, then table name.
Find the example's selected column in its database and inspect the first matching
table, alphabetically.
The DDL is schema context; executing SQL additionally requires the separately
obtained Spider/BIRD SQLite database files.

In [5]:
schemas = load_json(DATA_ROOT / DATASET / "schemas.json")
columns = load_json(DATA_ROOT / DATASET / "columns.json")
DB_ID = example["db_id"]
TARGET_COLUMN = example["selected_column"][0]
TABLE = next(table for table in sorted(columns[DB_ID]) if TARGET_COLUMN in columns[DB_ID][table])

print(f"Database: {DB_ID}; table: {TABLE}")
print(schemas[DB_ID][TABLE])
display(pd.DataFrame({"column": columns[DB_ID][TABLE]}))

Database: farm; table: farm
CREATE TABLE "farm" (
Farm_ID number,
Year number,
Total_Horses number,
Working_Horses number,
Total_Cattle number,
Oxen number,
Bulls number,
Cows number,
Pigs number,
Sheep_and_Goats number,
)


,column
0,Farm_ID
1,Year
2,Total_Horses
3,Working_Horses
4,Total_Cattle
5,Oxen
6,Bulls
7,Cows
8,Pigs
9,Sheep_and_Goats


### 5. Retrieve candidate columns without a model

BM25 (`sparse`) ranks schema columns within one database and table.
The first query uses an existing column name as a smoke test. Change `QUERY`
to a phrase to explore ranking. Scores are retrieval signals, not correctness labels.

Dense and hybrid retrieval are configured in `configs/*_hybrid.json` and the
[retrieval guide](../SCHEMA_RETRIEVAL.md); they additionally need an embedding model.

In [6]:
from clarity.retrieval.config import RetrievalConfig
from clarity.retrieval.factory import build_schema_index

QUERY = TARGET_COLUMN.replace("_", " ")
index = build_schema_index(
    RetrievalConfig(top_k=5, candidate_k=10),
    mode="sparse",
    schemas=schemas,
    columns=columns,
    base_directory=ROOT,
)
hits = index.search(QUERY, DB_ID, TABLE)
print(f"Query: {QUERY}")
display(pd.DataFrame(hits, columns=["table", "column", "data_type", "sparse_score", "score"]))

Query: Total Horses


,table,column,data_type,sparse_score,score
0,farm,Total_Horses,REAL,4.118055,0.016393
1,farm,Total_Cattle,REAL,2.059027,0.016129
2,farm,Working_Horses,REAL,2.059027,0.015873


### 6. Prepare a one-seed generation configuration

Start with the released lexical-column template. This example uses one seed,
one facet and one model per evaluator role to keep trial runs small. One seed
can still require several model calls and can yield no accepted conversations.
The preview and configuration validation below do not contact a model.

In [7]:
from clarity.generation.task_flows.components.data_types import ConversationGeneratorTask

task_config = copy.deepcopy(load_json(ROOT / "configs" / f"{DATASET}_lexical.json"))
task_config.update(max_examples=1, multifacet=1, batch_size=1)
for role in ("llm_retrieval", "llm_eval_term", "llm_eval_utt", "llm_sample_selectors"):
    task_config[role] = task_config[role][:1]
ConversationGeneratorTask.model_validate(task_config)

display(
    pd.DataFrame(
        [
            {"setting": name, "value": task_config[name]}
            for name in [
                "dataset",
                "ac_type",
                "retrieval_type",
                "similarity_criteria",
                "multifacet",
                "max_examples",
                "batch_size",
            ]
        ]
    ).set_index("setting")
)

,value
setting,
dataset,spider
ac_type,column_ambiguity
retrieval_type,llm
similarity_criteria,lexical
multifacet,1
max_examples,1
batch_size,1


### 7. Optionally generate new conversations

Set `CLARITY_MODEL` and `OPENAI_API_KEY` in your environment **before launching
Jupyter**; set `OPENAI_BASE_URL` if your compatible endpoint needs it. Keep actual
credentials out of notebook cells and configuration files.

To run, change `RUN_LIVE_GENERATION` to `True` in Setup and rerun the following
cells. Generation invokes your configured service and can incur charges. Each run
uses a fresh directory under `outputs/`, which Git and publication packaging exclude.
`data_root` resolves relative to the saved config file, so this cell adjusts it.

These are new experiments. They do not reproduce the paper's original model ensemble
or evaluation results. The seed controls local sampling, not model determinism.

In [8]:
receipt = None
generation_output = None
if RUN_LIVE_GENERATION:
    from clarity.cli import generate

    required = ("CLARITY_MODEL", "OPENAI_API_KEY")
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Set these environment variables first: " + ", ".join(missing))

    run_directory = ROOT / "outputs" / f"notebook-{uuid4().hex[:12]}"
    run_directory.mkdir(parents=True, exist_ok=False)
    run_config = copy.deepcopy(task_config)
    run_config["data_root"] = Path(os.path.relpath(DATA_ROOT, run_directory)).as_posix()
    config_path = run_directory / "config.json"
    with config_path.open("x", encoding="utf-8") as handle:
        json.dump(run_config, handle, indent=2)

    generation_output = run_directory / "conversations"
    receipt = generate(config_path, generation_output, seed=10, batch=False)
    display(pd.DataFrame([receipt]))
    if receipt["errors"]:
        raise RuntimeError("Generation recorded errors; inspect this run before using its outputs.")
else:
    print("Live generation skipped. Enable RUN_LIVE_GENERATION to run it.")

Live generation skipped. Enable RUN_LIVE_GENERATION to run it.


### 8. Inspect generated files

An enabled run writes intermediate `raw.jsonl`, scenario JSONL files, and
`receipt.json`. The receipt reports processed seeds, accepted conversations and
errors; acceptance is a pipeline outcome, not an independently verified quality score.
Review generated content and clear any sensitive outputs before sharing a notebook.

In [9]:
if generation_output is not None:
    print("Created files:", ", ".join(sorted(path.name for path in generation_output.iterdir())))
    generated = load_jsonl(generation_output / "concise_helpful.jsonl")
    if generated:
        display(pd.DataFrame(generated)[["db_id", "question", "usecase"]].head(3))
    else:
        print("No accepted concise-helpful conversations in this run.")
else:
    print("No generation files were created by this offline walkthrough.")

No generation files were created by this offline walkthrough.


## Checks

Confirm that both annotation files match their published data-manifest hashes,
the selected dialogue has supported roles, and retrieval stays within the
requested database/table. These checks remain offline.

In [10]:
data_manifest = load_json(ROOT / "provenance/data-manifest.json")
checks = []
for dataset in DATASETS:
    info = next(
        entry
        for entry in data_manifest["datasets"][dataset]["files"]
        if entry["path"].endswith("/annotations.jsonl")
    )
    with (ROOT / info["path"]).open("rb") as handle:
        matches = hashlib.file_digest(handle, "sha256").hexdigest() == info["sha256"]
    checks.append({"dataset": dataset, "annotation_hash_matches": matches})
assert all(row["annotation_hash_matches"] for row in checks)
assert set(schemas) == set(columns)
assert all(turn["role"] in {"USER", "AGENT"} for turn in example["conversation"])
assert hits and all(hit["db_id"] == DB_ID and hit["table"] == TABLE for hit in hits)
display(pd.DataFrame(checks).set_index("dataset"))
print("Offline checks passed.")

,annotation_hash_matches
dataset,
spider,True
bird,True


Offline checks passed.


## Next Steps

- Switch `DATASET` to `bird` and rerun the offline cells.
- Explore `configs/` for semantic ambiguity, confusion/unanswerability, value cases
  and hybrid retrieval. Value generation also needs the separately released value catalogs.
- Obtain upstream SQLite databases before attempting SQL execution. The inherited
  `expectedQuery` alone does not establish the correct terminal target of a conversation.
- Group experimental splits by source question and database. Conversation variants
  should not leak across train/test partitions; this release supplies no new official split.
- Preserve upstream attribution and data terms: [data license](../DATA_LICENSE.md),
  [notice](../NOTICE.md), [citation](../CITATION.cff).

The walkthrough covers the packaged files (12,098 Spider and 7,170 BIRD conversations),
not reconstructed paper evaluation splits. The BIRD paper count is 7,171; the release
retains and documents the one-record difference. See the [data card](../DATA_CARD.md)
and [verification notes](../VERIFICATION.md) for the limits of the released evidence.